<a href="https://colab.research.google.com/github/incrisvel/adaptaste/blob/main/parsing_ner_er.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install spacy networkx matplotlib
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 78.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
import spacy
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from spacy import displacy

nlp = spacy.load("pt_core_news_sm")

print(nlp.pipe_names)
print("spaCy:", spacy.__version__, "| Modelo:", nlp.meta["version"])

['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']
spaCy: 3.8.16 | Modelo: 3.8.0


## Dependências sintáticas e árvores

In [10]:
def analyze_dependency(text):
    doc = nlp(text)

    dependency_df = [
        {
            "token": token.text,
            "lemma": token.lemma_,
            "grammar_class": token.pos_,
            "dependency": token.dep_,
            "governance_token": token.head.text
        }
        for token in doc
    ]
    return doc, pd.DataFrame(dependency_df)

example_text = (
    "A comida estava deliciosa, "
    "mas o atendimento foi demorado."
)

doc, dependency_df = analyze_dependency(example_text)

display(dependency_df)

,token,lemma,grammar_class,dependency,governance_token
0,A,o,DET,det,comida
1,comida,comida,NOUN,nsubj,deliciosa
2,estava,estar,AUX,cop,deliciosa
3,deliciosa,delicioso,ADJ,ROOT,deliciosa
4,",",",",PUNCT,punct,demorado
5,mas,mas,CCONJ,cc,demorado
6,o,o,DET,det,atendimento
7,atendimento,atendimento,NOUN,nsubj:pass,demorado
8,foi,ser,AUX,aux:pass,demorado
9,demorado,demorar,VERB,conj,deliciosa


In [11]:
displacy.render(
    doc,
    style="dep",
    jupyter=True,
    options={"compact": True}
)

## Reconhecimento de entidades nomeadas (NER)

In [13]:
import pandas as pd
import json

# Abre arquivo de reviews
with open('reviews_preprocessed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

preprocessed_reviews_df = pd.DataFrame(
    review
    for review in data
)

preprocessed_reviews_df.head()

,review_id,review_date,place_id,review_text,rating,description_clean,description_tokenized,description_normalized,description_no_stopwords,description_stemmed,description_lemmatized,description_bigrams
0,Ci9DQUlRQUNvZENodHljRjlvT2xaVlRqSXpPRTFSYURKdl...,2026-07-30T17:09:02+00:00,0x6b0a883715bb4c6b:0,Excelente restaurante! A comida é muito saboro...,5.0,Excelente restaurante! A comida é muito saboro...,"[Excelente, restaurante, !, A, comida, é, muit...","[excelente, restaurante, a, comida, é, muito, ...","[excelente, restaurante, comida, muito, saboro...","[excel, restaurant, com, muit, sabor, bem, ser...","[excelente, restaurante, comer, muito, saboros...","[[excelente, restaurante], [restaurante, comer..."
1,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSl...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Fui pela primeira vez e gostei bastante! A com...,5.0,Fui pela primeira vez e gostei bastante! A com...,"[Fui, pela, primeira, vez, e, gostei, bastante...","[fui, pela, primeira, vez, e, gostei, bastante...","[primeira, vez, gostei, bastante, comida, muit...","[prim, vez, gost, bast, com, muit, sabor, peg,...","[primeiro, vez, gostar, bastante, comer, muito...","[[primeiro, vez], [vez, gostar], [gostar, bast..."
2,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONV...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Ambiente simples e gostoso familiar comida óti...,5.0,Ambiente simples e gostoso familiar comida óti...,"[Ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, gostoso, familiar, comida,...","[ambi, simpl, gost, famili, com, ótim, gost, m...","[ambiente, simples, gostoso, familiar, comer, ...","[[ambiente, simples], [simples, gostoso], [gos..."
3,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE,2022-08-30T17:10:53+00:00,0x6b0a883715bb4c6b:0,"Se é qualidade que procura, não tem outra igua...",5.0,"Se é qualidade que procura, não tem outra igua...","[Se, é, qualidade, que, procura, ,, não, tem, ...","[se, é, qualidade, que, procura, não, tem, out...","[qualidade, procura, não, outra, igual, agilid...","[qual, proc, não, outr, igual, agil, entreg, t...","[qualidade, procurar, não, outro, igual, agili...","[[qualidade, procurar], [procurar, não], [não,..."
4,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB,2022-08-30T17:10:52+00:00,0x6b0a883715bb4c6b:0,"Comida maravilhosa. Super caseira, bem tempera...",5.0,"Comida maravilhosa. Super caseira, bem tempera...","[Comida, maravilhosa, ., Super, caseira, ,, be...","[comida, maravilhosa, super, caseira, bem, tem...","[comida, maravilhosa, super, caseira, bem, tem...","[com, maravilh, sup, cas, bem, temper, tend, s...","[comida, maravilhoso, super, caseira, bem, tem...","[[comida, maravilhoso], [maravilhoso, super], ..."


In [ ]:
def extract_entities(texto):
    return pd.DataFrame([
        {
            "entity": ent.text,
            "category": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        }
        for ent in nlp(texto).ents
    ], columns=["entity", "category", "start", "end"])


In [19]:
# Extrai todos os textos
texts = preprocessed_reviews_df["review_text"].fillna("").astype(str).tolist()

docs = list(
    nlp.pipe(
        texts,
        batch_size=32
    )
)

entity_records = []

for row_index, doc in enumerate(docs):
    for ent in doc.ents:
        entity_records.append({
            "review_index": row_index,
            "review_text": preprocessed_reviews_df.iloc[row_index]["review_text"],
            "entity": ent.text,
            "entity_category": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char,
            "place_id": (
                preprocessed_reviews_df.iloc[row_index]["place_id"]
                if "place_id" in preprocessed_reviews_df.columns
                else None
            ),
            "review_id": (
                preprocessed_reviews_df.iloc[row_index]["review_id"]
            ),
        })

entities_df = pd.DataFrame(entity_records)

# Estiliza a exibição limitando o tamanho da coluna review_id
styled_df = entities_df.head(20).style.set_properties(
    **{
        'max-width': '100px',
        'overflow': 'hidden',
        'text-overflow': 'ellipsis',
        'white-space': 'nowrap'
    },
    subset=['review_id', 'place_id']
)

display(styled_df)

,review_index,review_text,entity,entity_category,start,end,place_id,review_id
0,1,"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Fui,PER,0,3,0x6b0a883715bb4c6b:0,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSllWZFpPVEZEUzNWdFdtYxAB
1,2,Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,Barra Velha,LOC,75,86,0x6b0a883715bb4c6b:0,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONVRWRTNUMmxwVTBOaVltYxAB
2,3,"Se é qualidade que procura, não tem outra igual. Agilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe. Parabéns!",Parabéns,MISC,149,157,0x6b0a883715bb4c6b:0,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE
3,4,"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",Super,MISC,20,25,0x6b0a883715bb4c6b:0,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB
4,4,"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",Super,MISC,133,138,0x6b0a883715bb4c6b:0,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB
5,4,"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",Blumenau,LOC,168,176,0x6b0a883715bb4c6b:0,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB
6,5,"As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo. A próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.",SENSACIONAL,ORG,51,62,0x6b0a883715bb4c6b:0,ChZDSUhNMG9nS0VJQ0FnSUNHc1pPSlNREAE
7,5,"As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo. A próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.",RECOMENDADÍSSIMO,MISC,140,156,0x6b0a883715bb4c6b:0,ChZDSUhNMG9nS0VJQ0FnSUNHc1pPSlNREAE
8,5,"As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo. A próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.",São Paulo,LOC,196,205,0x6b0a883715bb4c6b:0,ChZDSUhNMG9nS0VJQ0FnSUNHc1pPSlNREAE
9,5,"As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo. A próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.",Blumenau,LOC,249,257,0x6b0a883715bb4c6b:0,ChZDSUhNMG9nS0VJQ0FnSUNHc1pPSlNREAE


In [27]:
displacy.render(nlp(texts[5]), style="ent", jupyter=True)

## Extração de relações e SVO

In [31]:
def extract_svo(doc):
    relations = []

    for token in doc:
        # Considera verbos como possíveis núcleos da relação.
        if token.pos_ not in {"VERB", "AUX"}:
            continue

        subjects = [
            child for child in token.children
            if child.dep_ in {"nsubj", "nsubj:pass", "csubj"}
        ]

        objects = [
            child for child in token.children
            if child.dep_ in {
                "obj",
                "dobj",
                "iobj",
                "obl"
            }
        ]

        for subject in subjects:
            for obj in objects:
                relations.append({
                    "subject": subject.text,
                    "verb": token.lemma_,
                    "object": obj.text,
                    "sentence": token.sent.text
                })

    return relations

example_doc = nlp(texts[6])
svo_records = extract_svo(example_doc)

display(pd.DataFrame(svo_records))

,subject,verb,object,sentence
0,ninguém,atender,ligações,"A comida deve ser boa pelas indicações mas, o ..."
